This is an extremely simple FEP computation to get the idea of how FEP works.

We will calculate the free energy difference between two harmonic wells (A and B) using FEP.

In [ ]:
import numpy as np

# --- parameters ---
kB = 1.0
T = 1.0
beta = 1.0 / (kB * T)

k = 5.0
A = np.array([0.0, 0.0])
B = np.array([1.0, 0.0])

dt = 1e-3
nu = 1.0
D = kB * T * nu
n_steps = 40000
burn_in = 5000
stride = 10          # keep every 10th point after burn-in

# --- potentials ---
def U_A(x):
    return 0.5 * k * np.sum((x - A) ** 2)

def U_B(x):
    return 0.5 * k * np.sum((x - B) ** 2)

def grad_U_A(x):
    return k * (x - A)

In [ ]:

# --- Langevin dynamics in state A (lambda = 0) ---
x = np.array([0.0, 0.0])
traj = []
dUs = []

for step in range(n_steps):
    grad = grad_U_A(x)
    noise = np.sqrt(2.0 * D * dt) * np.random.randn(2)
    x = x - nu * grad * dt + noise

    if step >= burn_in and (step - burn_in) % stride == 0:
        traj.append(x.copy())
        dUs.append(U_B(x) - U_A(x))

traj = np.array(traj)
dUs = np.array(dUs)

 # if the overlap between states is good enough this works
 # otherwise, we need to use intermediate states (lambda windows)
 # and compute ΔF between adjacent states and sum them up.
DeltaF = -1.0 / beta * np.log(np.mean(np.exp(-beta * dUs)))
print(f"ΔF (A→B) ≈ {DeltaF:.3f}  (kBT units)")

In [ ]:
import plotly.graph_objects as go

# --- potential surface for plotting ---
xg = np.linspace(-1, 2, 80)
yg = np.linspace(-1, 1, 80)
X, Y = np.meshgrid(xg, yg)

U_A_grid = 0.5 * k * ((X - A[0])**2 + (Y - A[1])**2)
U_B_grid = 0.5 * k * ((X - B[0])**2 + (Y - B[1])**2)
U_min = np.minimum(U_A_grid, U_B_grid)

# z-values of trajectory on U_A surface
Z_traj = np.array([U_A(p) for p in traj])

# --- base figure: surface (0), wells (1), particle (2), tail (3) ---
fig = go.Figure(
    data=[
        # 0: energy surface
        go.Surface(
            x=X, y=Y, z=U_min,
            colorscale="RdBu_r",
            opacity=0.8,
            showscale=False,
            name="Energy surface"
        ),
        # 1: wells A and B
        go.Scatter3d(
            x=[A[0], B[0]],
            y=[A[1], B[1]],
            z=[0.0, 0.0],
            mode="markers+text",
            marker=dict(size=6, color=["blue", "red"]),
            text=["A", "B"],
            textposition="top center",
            name="Wells"
        ),
        # 2: particle at first frame
        go.Scatter3d(
            x=[traj[0, 0]],
            y=[traj[0, 1]],
            z=[Z_traj[0]],
            mode="markers",
            marker=dict(size=6, color="black"),
            name="Particle"
        ),
        # 3: trajectory tail (starts with single point)
        go.Scatter3d(
            x=traj[:1, 0],
            y=traj[:1, 1],
            z=Z_traj[:1],
            mode="lines",
            line=dict(color="black", width=4),
            name="Trajectory"
        ),
    ]
)

# --- build frames: only update traces 2 (particle) and 3 (tail) ---
frames = []
for i in range(len(traj)):
    frames.append(
        go.Frame(
            data=[
                go.Scatter3d(
                    x=[traj[i, 0]],
                    y=[traj[i, 1]],
                    z=[Z_traj[i]],
                    mode="markers",
                    marker=dict(size=6, color="black"),
                ),
                go.Scatter3d(
                    x=traj[:i+1, 0],
                    y=traj[:i+1, 1],
                    z=Z_traj[:i+1],
                    mode="lines",
                    line=dict(color="black", width=4),
                )
            ],
            traces=[2, 3],   # update only particle and tail
            name=f"frame{i}"
        )
    )

fig.frames = frames

# --- layout: fixed axis ranges + controls ---
fig.update_layout(
    title="Animated 2D Langevin Trajectory on Harmonic Wells",
    scene=dict(
        xaxis_title="x",
        yaxis_title="y",
        zaxis_title="U(x, y)",
        xaxis=dict(range=[-1, 2], autorange=False),
        yaxis=dict(range=[-1, 1], autorange=False),
        zaxis=dict(range=[0, 5], autorange=False),
    ),
    width=900,
    height=700,
    updatemenus=[
        dict(
            type="buttons",
            showactive=False,
            y=1.05,
            x=0.0,
            xanchor="left",
            yanchor="bottom",
            buttons=[
                dict(
                    label="Play",
                    method="animate",
                    args=[
                        None,
                        dict(
                            frame=dict(duration=40, redraw=True),
                            fromcurrent=True,
                            transition=dict(duration=0),
                        ),
                    ],
                ),
                dict(
                    label="Pause",
                    method="animate",
                    args=[
                        [None],
                        dict(
                            frame=dict(duration=0, redraw=False),
                            mode="immediate",
                            transition=dict(duration=0),
                        ),
                    ],
                ),
            ],
        )
    ],
    sliders=[
        dict(
            steps=[
                dict(
                    method="animate",
                    args=[
                        [f"frame{i}"],
                        dict(
                            mode="immediate",
                            frame=dict(duration=0, redraw=True),
                            transition=dict(duration=0),
                        ),
                    ],
                    label=str(i),
                )
                for i in range(len(frames))
            ],
            x=0.1,
            y=0,
            currentvalue=dict(prefix="frame: "),
            len=0.9,
        )
    ],
)

fig.show()